# 异常链和高级异常处理

Python 中异常链、上下文抑制等高级异常处理技术。

## 异常链基础

In [ ]:
# 异常链作用：
# 1. 保留原始异常信息
# 2. 提供完整的错误上下文
# 3. 便于问题的根因分析

# Python 异常链的三种形式：
# 1. 隐式链接 (__context__)
# 2. 显式链接 (__cause__)
# 3. 抑制链接 (from None)

## 隐式异常链

In [ ]:
# 当在异常处理过程中发生新异常时，Python 会自动创建隐式链接

try:
    result = 10 / 0  # ZeroDivisionError
except ZeroDivisionError:
    value = int('abc')  # ValueError - 在处理前一个异常时发生

# 可以通过 __context__ 访问原始异常

## 显式异常链 (raise...from)

In [ ]:
# 使用 raise...from 显式链接异常

def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        # 保留原始异常，添加更友好的错误信息
        raise ValueError('除数不能为零') from e

try:
    divide(10, 0)
except ValueError as e:
    print(f'捕获异常：{e}')
    print(f'原始原因：{e.__cause__}')

## 抑制异常链

In [ ]:
# 使用 from None 抑制异常链
# 适用于不想暴露内部实现细节的场景

class ConfigError(Exception):
    """配置错误"""
    pass

def load_config():
    try:
        # 可能抛出 FileNotFoundError 或 JSONDecodeError
        with open('config.json') as f:
            import json
            return json.load(f)
    except Exception:
        # 抑制原始异常，只抛出统一的配置错误
        raise ConfigError('配置文件加载失败') from None

try:
    load_config()
except ConfigError as e:
    print(f'配置错误：{e}')
    print(f'__cause__: {e.__cause__}')  # None

## else 和 finally

In [ ]:
# else: 没有异常时执行
# finally: 无论是否异常都执行

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f'错误：{e}')
        return None
    else:
        print('计算成功')
        return result
    finally:
        print('清理资源...')

print(safe_divide(10, 2))
print('---')
print(safe_divide(10, 0))

## 自定义异常

In [ ]:
# 自定义异常类
class ValidationError(Exception):
    """数据验证错误"""
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f'{field}: {message}')

def validate_age(age):
    if not isinstance(age, int):
        raise ValidationError('age', '必须是整数')
    if age < 0 or age > 150:
        raise ValidationError('age', '必须在 0-150 之间')
    return True

try:
    validate_age(-1)
except ValidationError as e:
    print(f'验证错误：{e}')
    print(f'字段：{e.field}')